# 02. Limpeza de vendas e faturamento mensal por vendedor

Meta: deixar `vendas.csv` pronto para calcular o faturamento mensal por vendedor de 2020 a 2024.
Regras na spec `specs/01-dados-e-limpeza.md` (RN-10 a RN-19). Nada e inventado: o que nao da para confiar vai para quarentena.

In [1]:
import pandas as pd
from pathlib import Path
from cristalux.config import settings
from cristalux.cleaning import parsers as p
from cristalux.cleaning.base import ler_csv
from cristalux.pipeline.run import ARQUIVOS, limpar_tudo, relatorio_qualidade

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200, "display.max_colwidth", 60)
raws, res = limpar_tudo(settings.data_dir)

## Antes

In [2]:
raws["vendas"].sample(12, random_state=3).drop(columns="linha_origem")

,id_venda,data,id_vendedor,id_comprador,produto,categoria,quantidade,valor_unitario,valor_total,desconto,status,regiao,observacoes
105,VD0047,31/11/2023,V016,C001,teto solar,Reparos,8,"R$ 2.478,34","R$ 19.826,72",N/A,CANCELADO,São Paulo,negociacao dificil
174,VD0052,22/11/2022,V004,C019,Kit Reparo Trinca,VIDROS,18,257.94,4053.73,Sim,CONCLUIDO,Rio Grande do Sul,frete gratis
78,VD0086,2024-09-18,V014,C019,VID. TRASEIRO,acessorios,-2,"R$ 283,28","R$ 566,56",10,CANCELADO,RJ - Capital,
3,VD0053,30/06/2023,V001,C030,vidro traseiro completo,REPAROS,20,1212.60,24252.00,5%,pendente,MG,
163,VD0013,2020-03-01,V012,C029,Moldura Parabrisas,vidros,15,4294.47,64417.05,Sim,Cancelada,GO,
95,VD0037,30/08/2021,V011,C020,parabrisas dianteiro,Reparos,-2,2378.05,-4756.10,Não,Fechado,SC,
151,VD0141,25/03/2023,V001,C013,kit reparo,Reparos,3,"R$ 3.016,52","R$ 12.508,83",nenhum,Em Aberto,Goiás,
6,VD0022,00/12/2022,V017,C030,vidro traseiro completo,acessorios,4,4361.7,17446.8,10,Concluída,sao paulo,frete gratis
98,VD0153,N/A,V003,C018,Pelicula Protetora,Acessórios,1,679.92,558.12,5%,pendente,Parana,urgente
183,VD0011,28/02/2022,V006,C015,Vidro Lateral Esq.,reparos,-2,3086.85,-8145.28,Não,Cancelada,Rio de Janeiro,negociacao dificil


## Resultado da limpeza

In [3]:
r = res["vendas"]
print(r.stats)
r.conferir_conservacao(len(raws["vendas"]))
print("conservacao ok:", len(r.clean), "limpas +", len(r.quarantine), "quarentena =", len(raws["vendas"]), "brutas")

{'linhas_brutas': 190, 'linhas_limpas': 96, 'quarentena': 94}
conservacao ok: 96 limpas + 94 quarentena = 190 brutas


## Motivos de quarentena

Uma linha pode ter mais de um motivo. `duplicata_exata` e `id_duplicado` sao removidas por repeticao, as demais por dado invalido.

In [4]:
motivos = r.quarantine["motivo"].str.split(";").explode().value_counts().rename("linhas").to_frame()
motivos["%_das_190"] = (100 * motivos["linhas"] / len(raws["vendas"])).round(1)
motivos

,linhas,%_das_190
motivo,,
data_vazia,26,13.7
data_inexistente,19,10.0
quantidade_negativa,19,10.0
duplicata_exata,12,6.3
quantidade_invalida,10,5.3
comprador_inexistente,9,4.7
data_fora_do_periodo,8,4.2
data_formato_desconhecido,4,2.1
valor_unitario_invalido,3,1.6


Decisoes e justificativas:

- **Data ausente ou impossivel (30/02, 31/11, mes 22, 2027, 2028):** quarentena. Sem data nao ha como alocar a venda a um mes, e inventar uma data distorceria a serie mensal.
- **Quantidade negativa:** quarentena como erro de lancamento. Devolucoes ja tem status proprio (`devolvida`), entao um valor negativo nao e a forma de registra-las.
- **Duplicatas:** linha identica mantem uma. Mesmo `id_venda` com conteudo diferente: fica o registro valido, mais completo e mais recente.
- **Comprador ou vendedor inexistente:** quarentena. Venda sem cadastro nao pode ser atribuida.

## Flags nas linhas que ficaram

In [5]:
flags = r.clean["flags"].explode().value_counts().rename("linhas").to_frame()
flags

,linhas
flags,
categoria_incoerente,72
desconto_indefinido,26
venda_antes_da_admissao,25
valor_total_divergente,12
data_ambigua,4


In [6]:
c = r.clean
print("Descontos acima de 10% (limite da decisao D018):", int(c["flags"].map(lambda f: "desconto_acima_limite" in f).sum()))
print("valor_total informado difere de quantidade x preco:", int(c["flags"].map(lambda f: "valor_total_divergente" in f).sum()))
c[c["flags"].map(lambda f: "valor_total_divergente" in f)][["id_venda", "quantidade", "valor_unitario", "desconto", "valor_total_informado", "valor_total", "valor_liquido"]]

Descontos acima de 10% (limite da decisao D018): 0
valor_total informado difere de quantidade x preco: 12


,id_venda,quantidade,valor_unitario,desconto,valor_total_informado,valor_total,valor_liquido
7,VD0016,1,1175.74,NaN,840.33,1175.74,1175.74
14,VD0041,18,2817.13,0.05,69287.86,50708.34,48172.92
19,VD0050,16,3010.46,0.10,48168.75,48167.36,43350.62
21,VD0052,18,257.94,NaN,4053.73,4642.92,4642.92
33,VD0064,6,281.18,0.00,1525.96,1687.08,1687.08
49,VD0088,12,1208.54,0.00,18320.16,14502.48,14502.48
53,VD0097,18,1884.67,NaN,46217.57,33924.06,33924.06
62,VD0108,10,1918.67,0.10,19162.43,19186.70,17268.03
75,VD0141,3,3016.52,0.00,12508.83,9049.56,9049.56
82,VD0151,9,2042.21,0.10,14637.50,18379.89,16541.90


Conferindo `valor_total` contra `quantidade x preco unitario`, o total informado bate exatamente em 84 das 96 linhas validas,
**inclusive nas vendas com desconto**: a origem nao aplica o desconto no total. Por isso `valor_total` (o faturamento) segue a origem,
e a coluna `valor_liquido` guarda o valor depois do desconto. As 12 linhas restantes divergem das duas formas
(flag `valor_total_divergente`) e usam o valor recalculado.

Desconto `Sim` sem percentual ou vazio vira nulo e e tratado como 0 no calculo do liquido, com o flag `desconto_indefinido`.

## Categoria informada nao e confiavel

In [7]:
pd.crosstab(c["produto"], c["categoria"])

categoria,Acessórios,Reparos,Sensores,Vidros
produto,,,,
Kit de Reparo,4,2,1,6
Moldura de Parabrisas,5,3,2,3
Parabrisas Dianteiro,1,0,2,2
Película Protetora,5,3,3,2
Sensor de Chuva,5,2,3,1
Teto Solar,2,4,4,3
Vidro Lateral,3,5,3,0
Vidro Lateral Direito,1,0,0,0
Vidro Lateral Esquerdo,0,0,1,2


A categoria escrita na venda nao tem relacao estavel com o produto (um sensor aparece como Reparos, um teto solar como Vidros).
Por isso a base limpa guarda tambem `categoria_produto` (a categoria natural do produto) e as views usam essa.

## Faturamento mensal por vendedor, 2020 a 2024

Somente vendas concluidas. As grafias `concluida`, `concluido`, `fechada` e `fechado` (com ou sem acento, em qualquer caixa) contam como concluida.

In [8]:
concl = c[c["status"] == "concluida"].copy()
concl["ano_mes"] = concl["data"].map(lambda d: d.strftime("%Y-%m"))
vend = res["vendedores"].clean.set_index("id_vendedor")["nome"]
concl["vendedor"] = concl["id_vendedor"].map(vend)
fat = concl.groupby(["ano_mes", "id_vendedor", "vendedor"])["valor_total"].sum().round(2).rename("faturamento").reset_index()
print(len(fat), "combinacoes mes x vendedor com faturamento")
fat

32 combinacoes mes x vendedor com faturamento


,ano_mes,id_vendedor,vendedor,faturamento
0,2020-01,V002,Bruno Alves,39884.68
1,2020-01,V003,Carla Mendonça,26349.31
2,2020-01,V007,Gabriela Rocha,48167.36
3,2020-01,V008,Henrique Souza,69765.00
4,2020-01,V012,Leonardo Bastos,9497.34
5,2020-01,V015,Olivia Cardoso,2235.30
6,2020-01,V017,Queila Barbosa,82347.20
7,2020-03,V007,Gabriela Rocha,75070.20
8,2020-03,V012,Leonardo Bastos,41639.12
9,2021-01,V017,Queila Barbosa,16618.64


In [9]:
pivo = fat.pivot_table(index="ano_mes", columns="vendedor", values="faturamento", aggfunc="sum").fillna(0)
pivo.round(0).astype(int)

vendedor,Ana Paula Ferreira,Bruno Alves,Carla Mendonça,Daniel Costa,Eliane Torres,Felipe Nunes,Gabriela Rocha,Henrique Souza,Isabela Martins,João Pedro Lima,Leonardo Bastos,Mariana Fonseca,Nicolas Andrade,Olivia Cardoso,Pedro Henrique,Queila Barbosa
ano_mes,,,,,,,,,,,,,,,,
2020-01,0,39885,26349,0,0,0,48167,69765,0,0,9497,0,0,2235,0,82347
2020-03,0,0,0,0,0,0,75070,0,0,0,41639,0,0,0,0,0
2021-01,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16619
2021-07,0,1176,19187,0,0,0,0,0,0,77776,0,17867,6717,0,0,0
2022-02,0,0,0,0,0,0,0,0,0,0,0,0,0,0,58199,0
2022-05,0,0,1179,0,0,0,60982,0,0,0,0,0,0,0,0,0
2022-11,0,50708,13007,4643,0,0,14687,0,0,0,0,0,0,0,0,0
2023-03,0,0,0,26901,0,0,0,0,0,0,0,0,0,0,0,0
2023-04,38049,0,0,0,0,0,0,10529,0,0,0,0,0,0,0,0


A tabela e esparsa: 33 vendas concluidas em 60 meses. Isso nao e erro da limpeza, e o que sobra dos dados
(metade das linhas foi rejeitada e entre as validas muitas estao canceladas, devolvidas ou pendentes).

## Quanto cada regra pesa no faturamento

In [10]:
total = lambda df: round(df["valor_total"].sum(), 2)
pd.Series({
    "so concluidas (regra adotada)": total(c[c["status"] == "concluida"]),
    "concluidas + pendentes": total(c[c["status"].isin(["concluida", "pendente"])]),
    "todas as vendas validas": total(c),
    "concluidas sem vendas antes da admissao": total(c[(c["status"] == "concluida") & ~c["flags"].map(lambda f: "venda_antes_da_admissao" in f)]),
}).rename("faturamento_total")

so concluidas (regra adotada)               974873.01
concluidas + pendentes                     1655685.95
todas as vendas validas                    2544968.51
concluidas sem vendas antes da admissao     489522.66
Name: faturamento_total, dtype: float64

## Conferencia com o banco

Se o Postgres estiver no ar, a view `vw_faturamento_mensal_vendedor` deve dar o mesmo total.

In [11]:
try:
    from cristalux.db.setup import conectar
    with conectar() as conn, conn.cursor() as cur:
        cur.execute("SELECT COALESCE(SUM(faturamento), 0), COUNT(*) FROM clean.vw_faturamento_mensal_vendedor")
        total_banco, linhas_banco = cur.fetchone()
    print("banco:", total_banco, linhas_banco, "| notebook:", round(fat["faturamento"].sum(), 2), len(fat))
except Exception as e:
    print("banco indisponivel:", type(e).__name__)

banco: 974873.01 32 | notebook: 974873.01 32
